# Serializing and Restoring Objects

CSC-239 · Module 11 · Lesson 2 of 2

You can save primitive fields under an agreed binary schema. This lesson saves the state of a supported Java object and checks the type and values of the object restored from that data.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Serialize and restore a supported object from a file created by the same complete example.
- Check the restored type before casting and distinguish saved fields from transient state.


## Why This Matters

An equipment card needs its item name and available count after it is restored. A temporary screen selection should begin fresh rather than becoming saved business state.


## Check Your Starting Point

Recall the difference between instance state and static class state. Explain why a reader must use the intended binary format and why streams should be closed before deleting a temporary file. Retrieve the distinction between an object and a reference to it.

**My explanation:**


## Concept

### Save state as an object representation

**Object serialization** writes a representation of an object's state into an object output stream. **Object deserialization** reconstructs an object from a compatible saved representation. These operations use ObjectOutputStream and ObjectInputStream.

The saved data is not a copy of your Java source code or a saved running kernel. The reader still needs compatible class definitions. Methods are supplied by the available class, while the representation supplies the saved state.

This lesson reads only files that each complete example created itself. A later type check does not make arbitrary serialized input safe to read. For the project, use the supplied classes and data exchange rules.

### Mark a class as serializable

Serializable is a **marker interface**: it has no methods that a class must implement. Adding implements Serializable declares support for Java serialization. This differs from the interfaces that required you to supply an operation in Module 6.

Default serialization saves ordinary instance state. Static class state is not part of that per-object representation. A later section explains how to exclude temporary instance fields. Required objects reached through saved fields must also support serialization. String supports serialization, so the String and primitive fields in these examples are suitable.

A **serialization version identifier** is the class's explicit serialVersionUID. Java uses it as part of class-version compatibility checks. The declaration private static final long serialVersionUID = 1L gives this example a fixed identifier. The L suffix makes the literal a long value, and final prevents reassignment.

Matching this identifier does not make every class change compatible. It also does not replace the supplied project's data exchange rules or field requirements. Keep the example definition unchanged for a round-trip test.

### Write an object, then read its result

writeObject accepts an object reference and writes the supported state. readObject returns a reference declared as Object, Java's common superclass type for class instances. A reference declared as Object does not expose every method of the more specific object it may refer to.

A **runtime type check** uses instanceof to ask whether a non-null value is compatible with a particular reference type. If that check passes, a **reference cast** requests that more specific type. The cast does not manufacture a different kind of object; it changes the type through which the same restored object is accessed. A cast to an incompatible reference type throws ClassCastException. This exception reports that the actual object cannot be used through the requested type.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("label-card-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("lab"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof LabelCard) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

This prints Label: lab. The parenthesized type in (LabelCard) value is the cast. It comes after the instanceof check. Calling a LabelCard method through a reference declared as Object would not satisfy Java's declared-type rules.

The objects are written and read through object streams. A DataInputStream from the previous lesson does not become the correct reader merely because both files have a .bin extension.

readObject can throw IOException for input problems and ClassNotFoundException when a required class cannot be found. These are checked exceptions. As in the previous lesson, conventional Java methods must handle or declare them; IJava permits these top-level demonstration calls.

### Exclude temporary instance state

A **transient instance field** is excluded from default serialization. In these ordinary serializable classes, default deserialization does not run the serializable class's constructor or its instance field initializers. A transient int therefore begins as zero, and a transient boolean begins as false, unless custom handling supplies another value.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class SelectionCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    private transient boolean selected;
    public SelectionCard(String label) {
        this.label = label;
        this.selected = true;
    }
    public String getLabel() { return label; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("selection-card-", ".bin");
try {
    SelectionCard original = new SelectionCard("kit");
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof SelectionCard) {
            SelectionCard restored = (SelectionCard) value;
            System.out.println(restored.getLabel());
            System.out.println("Original: " + original.isSelected());
            System.out.println("Restored: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

This prints kit, Original: true, and Restored: false. The label was saved. The transient selection was not, and the constructor's assignment to true was not repeated during restoration. The original object's selection remains true because reading a new restored object does not change the original.

Static fields also are not saved as each object's ordinary instance state. You already learned that they belong to the class. Do not expect restoring one object to restore a whole application's class-wide state.

Only exclude a field when the object's design can handle that field's restored default. A field that must retain a business value should not be marked transient merely to make a test pass.

### Check an unexpected stored type

A controlled test can write a String into an object stream, then ask whether the returned value is the card type your reader expects. The instanceof check should fail, and the reader should report that mismatch without attempting the cast.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class TypeCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public TypeCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("type-check-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("lab");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof TypeCard) {
            TypeCard restored = (TypeCard) value;
            System.out.println(restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

This prints Unexpected object type. The String is valid serialized data, but it is not the card type required by this reader. That differs from a damaged file or a missing class definition. If value were null, instanceof would also be false.

### Keep the notebook and project contracts clear

Each complete notebook fixture defines its class, creates its own file, writes and reads it within that kernel session, and deletes the file. IJava supports these round trips with the supplied Java kernel.

Do not use an IJava-generated class identity as a promised long-term file format or shared project data agreement. The Super Ghost project supplies compiled sharedCode classes. Preserve those definitions and implement the actual IOManager method declarations when creating MyIOManager.java. The tutorial card classes demonstrate the mechanism; they do not replace that assignment interface.

For each test, compare the saved field values, the restored transient defaults, and the actual result type. Include a valid zero count and a controlled wrong-type file. Recreate all required state instead of depending on a previous notebook's kernel or temporary files.


## Video Demonstration

Watch ScoreCard retain its owner and points while its transient view count starts fresh after restoration. Identify the type check that must pass before the cast.

<video controls preload="metadata" width="960">
  <source src="media/02_serializing_and_restoring_objects/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_serializing_and_restoring_objects/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the serializing and restoring objects demonstration transcript](media/02_serializing_and_restoring_objects/transcript.md).


## Worked Example

**Subgoal 1: define supported state.** Mark ScoreCard Serializable, give it a fixed version identifier, and mark views transient.

**Subgoal 2: complete the round trip.** Write the original object, close the writer, and read an Object from a new input stream.

**Subgoal 3: check before using specific methods.** Test the result type, cast, and compare saved fields with transient state.


In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Maya", 7);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}


Expected output:

```text
Maya: 7
Original views: 1
Restored views: 0
```

The restored card has Maya and seven points. The original constructor set its view count to one. That field is transient, so the restored count is zero under default deserialization. The type check succeeds before the program calls ScoreCard readers.


## Predict, Run, Trace, and Explain

### Predict saved fields and temporary state

Predict every printed line before running. Distinguish the values stored in the owner and points fields from the temporary views field. Explain whether restoration repeats the constructor’s assignment to views, whether original is changed, and which condition must pass before the cast.

My predicted lines:

Which instance values are saved:

How the restored views value begins:

Effect on original:

Condition that permits the cast:


In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}


Run the complete prediction program in the Workspace. Retain your original prediction, record every output line and explain any correction. Explain why the two view counts can differ although the restored owner and points match the saved values. Identify where the writer and reader close and where the temporary file is removed. Replay the whole program and explain why it supplies its own class definition and creates new objects and a new temporary file rather than needing the earlier file.

My original prediction:

Actual output:

What I confirmed or corrected:

Saved fields compared with temporary state:

Resource closing and cleanup:

Why the whole-program replay is self-contained:

### Trace the class agreement and guarded cast

For each listed declaration, state whether it is per-object state, whether default serialization saves it as instance state, and its value or role after restoration. Explain how implements Serializable differs from implementing an interface that requires a method. Explain the jobs of serialVersionUID, the long literal 1L, readObject, instanceof and the cast. Why does matching serialVersionUID alone not prove that every class change is compatible? Trace the value from readObject through the condition to a ScoreCard method call.

| Declaration | Per-object state? | Saved by default as instance state? | Restored value or role |
| --- | --- | --- | --- |
| owner | | | |
| points | | | |
| views | | | |
| serialVersionUID | | | |


Why the marker interface requires no new method:

Version declaration and its limits:

Declared result type, runtime check and cast:

Why the cast does not create a new card:

<details>
<summary>Show answer</summary>

The saved non-transient instance fields retain owner Iris and points 11. Constructing original sets its transient views field to 1. Default deserialization reconstructs a ScoreCard without replaying that class’s constructor or its instance field initializers, so restored views begins at the int default 0. Reading the restored object does not change original, whose views stays 1. readObject returns a reference declared as Object. The instanceof ScoreCard check succeeds, and the following cast lets the code use the ScoreCard reader methods. The cast does not create another card or transform its values. The writer closes before the reader opens; the reader closes before finally deletes the example’s own temporary file. A full replay creates new example objects and a new private temporary file. Its output is checked within that complete round trip; the file is not promised as a format that can be carried into a different kernel session. Serializable is a marker interface with no required methods. The String and primitive instance fields used here support serialization. The static final long serialVersionUID identifies the class version for part of compatibility checking; it is not ordinary saved per-object state and cannot guarantee that every change to the class is compatible. The L suffix gives 1L the long type, and final prevents reassignment. Object is the declared result type of readObject; the runtime check establishes whether the actual non-null value supports ScoreCard access.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 1
Restored views: 0
```

Common error: Assuming restoration repeats the serializable class’s constructor. Assuming transient means the original field is immediately cleared. Treating a cast as construction of a replacement object.

</details>


### Handle a controlled unexpected value

This complete program creates its own file and deliberately writes a String instead of the original card. Predict the branch and all output before running. Run it and retain your explanation of why the reader must skip the cast. Then replace only the writeObject argument with null in the complete program. Predict and run again. Explain why these two inputs follow the same branch for different reasons and why creating original does not determine what was written.

My String-file prediction and selected branch:

Actual output and explanation:

My null-file prediction:

Actual output and explanation:

Why neither value should enter the card cast:

Which writeObject argument determines the stored value:


In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The file contains the String status, so readObject returns a String reference declared as Object. That value is not compatible with ScoreCard. The instanceof condition is false, the cast and card print statements are skipped, and the else branch prints Unexpected object type. The data is a valid serialized String, but it is the wrong type for this reader. In the null comparison, readObject returns null; instanceof ScoreCard is again false, so the same branch runs without casting null or calling a card reader. Constructing original does not force the writer to store it; the actual argument to writeObject determines what this file contains.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

Common error: Assuming the type of original controls the file even when writeObject receives a different argument. Casting before checking the actual restored value. Treating an unexpected stored type as proof that the file bytes are damaged.

**Check case 2.** A null value fails instanceof ScoreCard. The reader follows its mismatch branch without attempting the cast or calling a card method. All file creation, stream closing and cleanup remain inside this complete fixture.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(null);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete support, version and type checks

Replace MARKER_INTERFACE, VERSION_ID, RESTORE_OPERATION and EXPECTED_TYPE. Use Serializable, 1L, readObject and ScoreCard in their matching roles. Replace both appearances of EXPECTED_TYPE consistently. Copy the entire completed program into the work cell, predict and run it, and explain the check-before-cast order. State why the fixed version identifier is part of compatibility checking rather than a guarantee for every class change.

This sample is for repair:

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements MARKER_INTERFACE {
    private static final long serialVersionUID = VERSION_ID;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.RESTORE_OPERATION();
        if (value instanceof EXPECTED_TYPE) {
            ScoreCard restored = (EXPECTED_TYPE) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```


My four replacements and their roles:

Predicted output:

Actual output and post-run explanation:

Why the check and cast name the same type:

What the version identifier does and does not guarantee:

<details>
<summary>Show answer</summary>

Use Serializable in implements, 1L for the version identifier, readObject for the input operation and ScoreCard in both the runtime check and cast. These last two uses must refer to the expected card type. The check belongs before the cast so an incompatible value takes the alternate branch. The saved non-transient instance fields retain owner Iris and points 11. Constructing original sets its transient views field to 1. Default deserialization reconstructs a ScoreCard without replaying that class’s constructor or its instance field initializers, so restored views begins at the int default 0. Reading the restored object does not change original, whose views stays 1. readObject returns a reference declared as Object. The instanceof ScoreCard check succeeds, and the following cast lets the code use the ScoreCard reader methods. The cast does not create another card or transform its values. The writer closes before the reader opens; the reader closes before finally deletes the example’s own temporary file.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 1
Restored views: 0
```

Common error: Using different target types in the check and cast. Replacing the object input operation with a primitive read from the previous lesson. Treating the marker interface as a replacement for the class’s own reader methods.

</details>


### Change initial temporary state and then save it

Run the unchanged starter first. Change only the constructor’s views assignment from 1 to 5, retaining transient on that field. Predict all output, run the complete program and explain the original and restored view counts. Then compare a separate complete version that also removes transient from views. Predict and run again. Explain why that declaration changes which value survives restoration. Which declaration would fit a temporary view count, and which would fit a count that the design requires saving?


In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}


Unchanged starter output:

Prediction with constructor views 5 and transient retained:

Actual output and explanation:

Prediction with transient removed:

Actual output and explanation:

Which declaration fits the intended lifetime of this field:

<details>
<summary>Show answer</summary>

Changing the constructor sets original views to 5, but the field remains transient in the first modification. The restored count still begins at 0 because default deserialization does not replay this serializable class’s constructor assignment. Owner Iris and points 11 remain saved. In the separate comparison, removing transient makes views an ordinary non-static instance field, so the original value 5 is saved and restored. The result then has Original views: 5 and Restored views: 5. Choose the declaration from the intended lifetime of the data; do not discard a needed value merely to force a desired output.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 5;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 5
Restored views: 0
```

Common error: Expecting the changed constructor assignment to initialize the transient field during restoration. Assuming removing transient changes the original object’s already assigned value. Excluding a field that the design requires restoring.

**Additional test: `Constructor sets views 5; views is no longer transient`.** The complete fresh fixture defines views as an ordinary instance field before creating and writing its object. Its value 5 is therefore restored with owner and points. The earlier transient fixture remains a separate complete comparison.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 5;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 5
Restored views: 5
```

</details>


### Repair a check that permits the wrong cast

The displayed draft writes a String and then uses an intentionally incorrect type guard before its ScoreCard cast. ClassCastException reports an attempted cast to an incompatible reference type. Predict which operation fails and whether any card lines can print. Also predict what this same wrong condition would do if writeObject received original instead. Repair the condition to check the type required by the cast. Copy and run the complete repaired String-file program. Then change its writeObject argument back to original, predict and run the complete card-file version. Explain how the pair of tests prevents a reader from rejecting every input or casting an incompatible value.

This sample is for repair:

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof String) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```


My prediction for the String-file draft:

First failing operation and any earlier output:

My prediction for a ScoreCard under the same wrong guard:

My repaired condition:

Actual corrected String-file output:

Predicted and actual corrected card-file output:

Why both inputs are needed to check this repair:

<details>
<summary>Show answer</summary>

With the String-file draft, the wrong instanceof String condition succeeds, but the following ScoreCard cast cannot access that String as a ScoreCard. It raises ClassCastException before the first card print statement, so no named card lines are printed. With writeObject(original) and the same wrong String guard, the actual ScoreCard fails that condition and the program instead prints Unexpected object type. The guard is wrong for both cases. Repair it to instanceof ScoreCard before the unchanged ScoreCard cast. The corrected String-file program safely prints `Unexpected object type.`; the corrected card-file program prints Iris: 11, Original views: 1 and Restored views: 0. Both complete fixtures close their streams and delete their own files. A type check only helps when it checks the type required by the operation that follows.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

Common error: Checking for String before casting to ScoreCard. Moving the cast outside the guarded branch. Considering a reader correct merely because it rejects every input.

**Additional test: `After repairing the guard, write original ScoreCard instead of String`.** The correct guard accepts the restored ScoreCard and permits its cast and reader methods. This complements rejecting the String; an always-false condition could not pass both tests.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 1
Restored views: 0
```

</details>


## Independent Practice

### Save an equipment card and restore its lasting state

Define EquipmentCard implementing Serializable with `private static final long serialVersionUID = 1L`, private String item and int available fields, and a private transient boolean selected field. Its constructor accepts item and available, assigns those fields and sets selected true. Supply getItem, getAvailable and isSelected readers. Create a camera card with 3 available. Serialize it to a new private temporary file, read an Object, check instanceof EquipmentCard and cast only after that check. Print `camera: 3`, `Original selected: true` and `Restored selected: false`; use `Unexpected object type.` for a mismatching value. Include every import and all setup, close both streams and delete the file. Predict, run and explain the saved fields, temporary default and guard before testing variants.

My saved and temporary fields:

My prediction:

My complete Java program is in the work cell.

Actual output and post-run explanation:

Where the type check permits the cast:

Where resource closing and file cleanup occur:


### Test changed values and an unexpected stored type

Keep the baseline program. For each test, create and delete a new file within the complete program. First change only camera to projector with 3 available. Then use camera with zero available. Finally restore the baseline original card but change the writeObject argument to the String `"status"`; keep the EquipmentCard check and cast in their correct positions. Predict every output before each run and record your own explanation afterward. Identify which tests restore a supported card and which test must skip the cast. Explain why a true selection in original does not require a true selection in the restored card.

| Test | Prediction | Actual output | My explanation |
| --- | --- | --- | --- |
| camera, 3 available | | | |
| projector, 3 available | | | |
| camera, 0 available | | | |
| Write String status instead of the card | | | |


Which case must skip the cast and why:

Why saved values and transient defaults differ:

Resource closing and cleanup for each complete test:


<details>
<summary>Show answer</summary>

EquipmentCard implements the Serializable marker interface and declares the fixed static final long version identifier. Its String item and int available are ordinary instance fields. They restore camera and 3. The constructor sets original selected to true; selected is transient, so default deserialization does not save it or replay that constructor assignment and restored selected begins at false. The readObject result is held as Object, checked with instanceof EquipmentCard and cast only inside the matching branch. Reading this restored object does not clear selected in original. Both streams close and finally removes the complete example’s private temporary file. The different item and zero availability are ordinary saved-field values; changing them does not change the transient default. The controlled String test stores a valid serialized String instead of the card. It reaches the mismatch branch and prints `Unexpected object type.` without casting or calling EquipmentCard readers. This test checks the actual stored type rather than the class of an unrelated original variable. All tests use only files created within their own complete programs.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class EquipmentCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String item;
    private int available;
    private transient boolean selected;
    public EquipmentCard(String item, int available) {
        this.item = item;
        this.available = available;
        this.selected = true;
    }
    public String getItem() { return item; }
    public int getAvailable() { return available; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("equipment-card-", ".bin");
try {
    EquipmentCard original = new EquipmentCard("camera", 3);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof EquipmentCard) {
            EquipmentCard restored = (EquipmentCard) value;
            System.out.println(restored.getItem() + ": " + restored.getAvailable());
            System.out.println("Original selected: " + original.isSelected());
            System.out.println("Restored selected: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
camera: 3
Original selected: true
Restored selected: false
```

Common error: Leaving selected as ordinary saved state despite the temporary-field requirement. Assuming the constructor is replayed to set the restored selection true. Depending on a prior notebook class or temporary file.

**Additional test: Different item projector, available 3.** The different String is saved and restored; the transient boolean still begins false in the reconstructed card.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class EquipmentCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String item;
    private int available;
    private transient boolean selected;
    public EquipmentCard(String item, int available) {
        this.item = item;
        this.available = available;
        this.selected = true;
    }
    public String getItem() { return item; }
    public int getAvailable() { return available; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("equipment-card-", ".bin");
try {
    EquipmentCard original = new EquipmentCard("projector", 3);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof EquipmentCard) {
            EquipmentCard restored = (EquipmentCard) value;
            System.out.println(restored.getItem() + ": " + restored.getAvailable());
            System.out.println("Original selected: " + original.isSelected());
            System.out.println("Restored selected: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
projector: 3
Original selected: true
Restored selected: false
```

**Additional test: Camera with zero available.** Zero is a valid saved int value and remains zero after restoration. It does not change the separate transient selection rule.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class EquipmentCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String item;
    private int available;
    private transient boolean selected;
    public EquipmentCard(String item, int available) {
        this.item = item;
        this.available = available;
        this.selected = true;
    }
    public String getItem() { return item; }
    public int getAvailable() { return available; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("equipment-card-", ".bin");
try {
    EquipmentCard original = new EquipmentCard("camera", 0);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof EquipmentCard) {
            EquipmentCard restored = (EquipmentCard) value;
            System.out.println(restored.getItem() + ": " + restored.getAvailable());
            System.out.println("Original selected: " + original.isSelected());
            System.out.println("Restored selected: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
camera: 0
Original selected: true
Restored selected: false
```

**Additional test: Baseline original card exists, but writeObject receives String status.** The actual saved value is a String. The EquipmentCard guard fails and the cast is skipped, regardless of the separately created original card.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class EquipmentCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String item;
    private int available;
    private transient boolean selected;
    public EquipmentCard(String item, int available) {
        this.item = item;
        this.available = available;
        this.selected = true;
    }
    public String getItem() { return item; }
    public int getAvailable() { return available; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("equipment-card-", ".bin");
try {
    EquipmentCard original = new EquipmentCard("camera", 3);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof EquipmentCard) {
            EquipmentCard restored = (EquipmentCard) value;
            System.out.println(restored.getItem() + ": " + restored.getAvailable());
            System.out.println("Original selected: " + original.isSelected());
            System.out.println("Restored selected: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

</details>


## Summary

Serialization saves a supported object representation, and deserialization reconstructs compatible state. Serializable marks support; serialVersionUID participates in compatibility checks. readObject returns Object, so verify the expected type before casting. Default serialization excludes transient instance fields and static class state. The ordinary serializable constructor is not replayed to initialize a transient field.

Close the answers and explain why a restored object’s saved values and temporary values can differ.


## Reflection

Choose an object in your field with both lasting information and temporary interface state. Identify which values should survive restoration, which can begin fresh, and how your reader should respond to the wrong restored type.

**My design and explanation:**

Next, JavaFX will present object state through a graphical interface in Workspace Desktop.


## Supplemental Reading

- [Serializable and class compatibility](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/Serializable.html) defines the marker interface and version identifier.
- [ObjectOutputStream](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/ObjectOutputStream.html) explains which object fields are written.
- [ObjectInputStream](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/ObjectInputStream.html) explains restoration, result types, and exceptions.
- [ClassCastException](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/ClassCastException.html) describes an incompatible reference cast.
- [Serialization architecture](https://docs.oracle.com/en/java/javase/21/docs/specs/serialization/serial-arch.html) describes saved state and transient-field behavior.
